<div style="background-color:#EAF4EC; padding:20px; border-radius:10px;">

<h2 style="color:#2F6F4E; text-align:center; margin-bottom:5px;">
Forecasting: XGBoost
</h2>

<h4 style="color:#2F6F4E; text-align:center; margin-top:0;">
Master Thesis – ESG Governance Indicators (EU-27)
</h4>

<p style="font-size:14px; color:#2F6F4E;">
This notebook implements the <strong>XGBoost forecasting model</strong> as part of the CRISP-ML(Q) methodology.
The objective is to assess the performance of a <strong>non-linear, tree-based gradient boosting model</strong>
for forecasting ESG governance indicators across the EU-27, using the same temporally consistent
train–validation–test split defined in the econometric baseline.
</p>

<p style="font-size:14px; color:#2F6F4E;">
The model leverages lagged values, rolling statistics, and trend-based features to capture
both short-term persistence and non-linear temporal dynamics that are not addressed by linear
panel regression models.
</p>

<p style="font-size:14px; color:#2F6F4E;">
Forecasting accuracy is evaluated using out-of-sample performance metrics (MAE, RMSE, and R²),
and results are directly compared with the fixed effects panel regression baseline and other
machine learning approaches to identify indicator-specific gains from non-linear modeling.
</p>

</div>


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Import correct dataset

</div>

In [2]:
DATA_PATH = Path("../data/processed/data_final_governance_EU27.csv")
df_final = pd.read_csv(DATA_PATH)
print(df_final.shape)
df_final.head(1)

(405, 28)


,Country Name,Country Code,Indicator Name,Indicator Code,2000,2001,2002,2003,2004,2005,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,1.751059,1.751059,1.915159,1.963869,2.026624,1.912423,...,1.466502,1.472044,1.496881,1.50082,1.568587,1.521324,1.477709,1.242727,1.258587,1.133653


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Use the best XGBoost Hyperparameters acording to Optuna

</div>

In [3]:
# Hyperparameters (os teus)
xgb_parameters = {
    "n_estimators": 1502,
    "learning_rate": 0.01406430106815356,
    "max_depth": 2,
    "min_child_weight": 8,
    "subsample": 0.6509501606637498,
    "colsample_bytree": 0.6795541596326934,
    "reg_alpha": 0.0004928380438312149,
    "reg_lambda": 0.210226460490979,
    "objective": "reg:squarederror",
    "random_state": 42,
    "tree_method": "hist",
    "n_jobs": -1
}

ROLL = 3
TRAIN_END = 2018
VAL_END = 2021
FORECAST_START = 2024
FORECAST_END = 2030

FEATURES = ["year", "lag_1", "lag_2", "lag_3", "rolling_mean_3", "rolling_std_3", "trend"]


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Convert df_final in df_long 

</div>

In [ ]:
df = df_final.copy()

# detectar colunas ano (strings tipo "2000", "2001", ...)
year_cols = [c for c in df.columns if str(c).isdigit()]
id_cols = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

df_long = df.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year",
    value_name="value"
)

df_long["year"] = pd.to_numeric(df_long["year"], errors="coerce").astype(int)
df_long["value"] = pd.to_numeric(df_long["value"], errors="coerce")

# manter só 2000–2023 (observado)
df_long = df_long[df_long["year"].between(2000, 2023)].copy()

# remover linhas sem valor
df_long = df_long.dropna(subset=["value"]).copy()

df_long.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,value
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Define the metrics and temporal features 

</div>

In [5]:
def eval_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

In [6]:
def add_time_features(df_in, roll=3):
    df = df_in.sort_values(["Country Code", "year"]).copy()

    # lags
    for lag in [1, 2, 3]:
        df[f"lag_{lag}"] = df.groupby("Country Code")["value"].shift(lag)

    # rolling (past only)
    g = df.groupby("Country Code")["value"]
    df["rolling_mean_3"] = g.transform(lambda s: s.shift(1).rolling(roll).mean())
    df["rolling_std_3"]  = g.transform(lambda s: s.shift(1).rolling(roll).std())

    # trend
    df["trend"] = df["lag_1"] - df["lag_2"]

    return df


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Train and evaluate model (1 indicator at the time)

</div>

In [7]:
indicator_codes = sorted(df_long["Indicator Code"].unique())
results = []
models  = {}

for code in indicator_codes:
    df_ind = df_long[df_long["Indicator Code"] == code].copy()
    if df_ind.empty:
        continue

    ind_name = df_ind["Indicator Name"].iloc[0]  # ← dentro do loop
    df_ind   = add_time_features(df_ind, roll=ROLL)

    req      = FEATURES + ["value"]
    df_model = df_ind.dropna(subset=req).copy()
    if df_model.empty:
        continue

    train_df = df_model[df_model["year"] <= TRAIN_END].copy()
    val_df   = df_model[(df_model["year"] > TRAIN_END) & (df_model["year"] <= VAL_END)].copy()
    test_df  = df_model[df_model["year"] > VAL_END].copy()

    if train_df.empty or val_df.empty or test_df.empty:
        continue

    model = XGBRegressor(**xgb_parameters)
    model.fit(train_df[FEATURES], train_df["value"])

    val_pred  = model.predict(val_df[FEATURES])
    test_pred = model.predict(test_df[FEATURES])

    mae_val,  rmse_val,  r2_val  = eval_metrics(val_df["value"],  val_pred)
    mae_test, rmse_test, r2_test = eval_metrics(test_df["value"], test_pred)

    models[code] = model
    results.append({
        "Indicator Code": code,
        "MAE_Val":    mae_val,
        "RMSE_Val":   rmse_val,
        "R2_Val":     r2_val,
        "MAE_Test":   mae_test,
        "RMSE_Test":  rmse_test,
        "R2_Test":    r2_test,
        "Train_Years": f"{train_df['year'].min()}-{train_df['year'].max()}",
        "Val_Years":   f"{val_df['year'].min()}-{val_df['year'].max()}",
        "Test_Years":  f"{test_df['year'].min()}-{test_df['year'].max()}",
        "N_Train":   len(train_df),
        "N_Val":     len(val_df),
        "N_Test":    len(test_df),
        "Countries": df_ind["Country Code"].nunique()
    })
    print(f" {code} | MAE_Test: {mae_test:.4f} | RMSE_Test: {rmse_test:.4f} | R2_Test: {r2_test:.4f}")

 CC.EST | MAE_Test: 0.0651 | RMSE_Test: 0.0820 | R2_Test: 0.9880
 GB.XPD.RSDV.GD.ZS | MAE_Test: 0.0514 | RMSE_Test: 0.0733 | R2_Test: 0.9934
 GE.EST | MAE_Test: 0.0943 | RMSE_Test: 0.1313 | R2_Test: 0.9454
 IP.JRN.ARTC.SC | MAE_Test: 1595.7327 | RMSE_Test: 2781.8114 | R2_Test: 0.9905
 IP.PAT.RESD | MAE_Test: 314.5060 | RMSE_Test: 957.0221 | R2_Test: 0.9851
 IT.NET.USER.ZS | MAE_Test: 1.3014 | RMSE_Test: 1.9451 | R2_Test: 0.8411
 NY.GDP.MKTP.KD.ZG | MAE_Test: 3.7427 | RMSE_Test: 4.3439 | R2_Test: -1.5465
 PV.EST | MAE_Test: 0.0823 | RMSE_Test: 0.1016 | R2_Test: 0.7789
 RL.EST | MAE_Test: 0.0531 | RMSE_Test: 0.0718 | R2_Test: 0.9837
 RQ.EST | MAE_Test: 0.0755 | RMSE_Test: 0.0966 | R2_Test: 0.9627
 SE.ENR.PRSC.FM.ZS | MAE_Test: 0.0064 | RMSE_Test: 0.0079 | R2_Test: 0.9173
 SG.GEN.PARL.ZS | MAE_Test: 1.6628 | RMSE_Test: 2.8960 | R2_Test: 0.9014
 SL.TLF.CACT.FM.ZS | MAE_Test: 0.7624 | RMSE_Test: 0.9811 | R2_Test: 0.9672
 SM.POP.NETM | MAE_Test: 113319.9834 | RMSE_Test: 213808.2892 | R2_Test

In [8]:
results_xgb = pd.DataFrame(results).sort_values("RMSE_Test").reset_index(drop=True)
results_xgb

,Indicator Code,MAE_Val,RMSE_Val,R2_Val,MAE_Test,RMSE_Test,R2_Test,Train_Years,Val_Years,Test_Years,N_Train,N_Val,N_Test,Countries
0,SE.ENR.PRSC.FM.ZS,0.004561,0.007295,0.929178,0.006428,0.007949,0.917292,2003-2018,2019-2021,2022-2023,432,81,54,27
1,VA.EST,0.052168,0.069593,0.961809,0.055027,0.066422,0.967541,2003-2018,2019-2021,2022-2023,432,81,54,27
2,RL.EST,0.054233,0.070447,0.984927,0.053139,0.071830,0.983719,2003-2018,2019-2021,2022-2023,432,81,54,27
3,GB.XPD.RSDV.GD.ZS,0.071416,0.098067,0.987827,0.051356,0.073265,0.993357,2003-2018,2019-2021,2022-2023,432,81,54,27
4,CC.EST,0.089825,0.118246,0.975812,0.065055,0.082008,0.987956,2003-2018,2019-2021,2022-2023,432,81,54,27
5,RQ.EST,0.104264,0.145249,0.903768,0.075517,0.096613,0.962702,2003-2018,2019-2021,2022-2023,432,81,54,27
6,PV.EST,0.078608,0.105662,0.838370,0.082266,0.101618,0.778925,2003-2018,2019-2021,2022-2023,432,81,54,27
7,GE.EST,0.079845,0.100606,0.968480,0.094302,0.131265,0.945433,2003-2018,2019-2021,2022-2023,432,81,54,27
8,SL.TLF.CACT.FM.ZS,0.753828,1.009903,0.965968,0.762391,0.981059,0.967188,2003-2018,2019-2021,2022-2023,432,81,54,27
9,IT.NET.USER.ZS,1.698069,2.155302,0.911682,1.301426,1.945054,0.841135,2003-2018,2019-2021,2022-2023,432,81,54,27


<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Recurcive Forecast: 2024 to 2030

</div>

In [9]:
forecast_rows = []

for code in indicator_codes:
    if code not in models:
        continue

    model = models[code]
    df_ind = df_long[df_long["Indicator Code"] == code].copy()
    ind_name = df_ind["Indicator Name"].iloc[0]

    for (ccode, cname), g in df_ind.groupby(["Country Code", "Country Name"]):
        g = g.sort_values("year").copy()

        # precisamos de pelo menos 4 valores para lag_1..lag_3
        if len(g) < 4:
            continue

        hist_years = g["year"].tolist()
        hist_vals = g["value"].astype(float).tolist()

        last_year = int(max(hist_years))
        start_year = max(FORECAST_START, last_year + 1)

        # se já não há nada para prever
        if start_year > FORECAST_END:
            continue

        # vamos estender a série com previsões (recursivo)
        hist_extended = hist_vals.copy()

        for y in range(start_year, FORECAST_END + 1):
            # criar features a partir da história estendida
            lag_1 = hist_extended[-1]
            lag_2 = hist_extended[-2]
            lag_3 = hist_extended[-3]

            last3 = hist_extended[-ROLL:]
            rolling_mean_3 = float(np.mean(last3))
            rolling_std_3 = float(np.std(last3, ddof=1)) if len(last3) >= 2 else 0.0
            trend = lag_1 - lag_2

            X_next = pd.DataFrame([{
                "year": int(y),
                "lag_1": float(lag_1),
                "lag_2": float(lag_2),
                "lag_3": float(lag_3),
                "rolling_mean_3": rolling_mean_3,
                "rolling_std_3": rolling_std_3,
                "trend": float(trend),
            }])

            y_hat = float(model.predict(X_next[FEATURES])[0])

            hist_extended.append(y_hat)

            forecast_rows.append({
                "Country Name": cname,
                "Country Code": ccode,
                "Indicator Name": ind_name,
                "Indicator Code": code,
                "year": int(y),
                "pred_xgb": y_hat,
                "type": "forecast"
            })

df_forecast = pd.DataFrame(forecast_rows)
df_forecast.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,pred_xgb,type
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2024,1.195147,forecast


In [10]:
df_observed = df_long.copy()
df_observed = df_observed.rename(columns={"value": "pred_xgb"})
df_observed["type"] = "observed"

df_observed = df_observed[[
    "Country Name","Country Code","Indicator Name","Indicator Code","year","pred_xgb","type"
]]

df_forecast = df_forecast[[
    "Country Name","Country Code","Indicator Name","Indicator Code","year","pred_xgb","type"
]]

# Juntar
df_xgb_full = pd.concat([df_observed, df_forecast], ignore_index=True)

# Garantir: observed 2000–2023 e forecast 2024–2030
df_xgb_full = df_xgb_full[
    ((df_xgb_full["type"] == "observed") & (df_xgb_full["year"].between(2000, 2023))) |
    ((df_xgb_full["type"] == "forecast") & (df_xgb_full["year"].between(2024, 2030)))
].copy()

# Garantir zero duplicados por (country, indicator, year)
df_xgb_full = (
    df_xgb_full.sort_values(["Country Code","Indicator Code","year","type"])
    .drop_duplicates(subset=["Country Code","Indicator Code","year"], keep="last")
    .reset_index(drop=True)
)

df_xgb_full.head(1)

,Country Name,Country Code,Indicator Name,Indicator Code,year,pred_xgb,type
0,Austria,AUT,Control of Corruption: Estimate,CC.EST,2000,1.751059,observed


In [11]:
print("Years observed:", df_xgb_full[df_xgb_full["type"]=="observed"]["year"].min(), "-", df_xgb_full[df_xgb_full["type"]=="observed"]["year"].max())
print("Years forecast:", df_xgb_full[df_xgb_full["type"]=="forecast"]["year"].min(), "-", df_xgb_full[df_xgb_full["type"]=="forecast"]["year"].max())

dups = df_xgb_full.duplicated(subset=["Country Code","Indicator Code","year"]).sum()
print("Duplicates:", dups)

print("Forecast coverage in 2030 (countries per indicator):")
print(df_xgb_full[df_xgb_full["year"]==2030].groupby("Indicator Code")["Country Code"].nunique().sort_values())

Years observed: 2000 - 2023
Years forecast: 2024 - 2030
Duplicates: 0
Forecast coverage in 2030 (countries per indicator):
Indicator Code
CC.EST               27
GB.XPD.RSDV.GD.ZS    27
GE.EST               27
IP.JRN.ARTC.SC       27
IP.PAT.RESD          27
IT.NET.USER.ZS       27
NY.GDP.MKTP.KD.ZG    27
PV.EST               27
RL.EST               27
RQ.EST               27
SE.ENR.PRSC.FM.ZS    27
SG.GEN.PARL.ZS       27
SL.TLF.CACT.FM.ZS    27
SM.POP.NETM          27
VA.EST               27
Name: Country Code, dtype: int64


## XGBoost Results — Interpretation

### Group 1 — Excellent (R²_Test > 0.90)

| Indicator Code | RMSE_Test | R²_Test |
|---|---|---|
| SE.ENR.PRSC.FM.ZS | 0.0079 | **0.9173** |
| VA.EST | 0.0664 | **0.9675** |
| RL.EST | 0.0718 | **0.9837** |
| CC.EST | 0.0820 | **0.9880** |
| RQ.EST | 0.0966 | **0.9627** |
| GB.XPD.RSDV.GD.ZS | 0.0733 | **0.9934** |
| IP.PAT.RESD | 957.02 | **0.9851** |
| IP.JRN.ARTC.SC | 2781.81 | **0.9905** |

XGBoost captures the temporal dynamics of these indicators very well — R² close to 1 in almost all cases, outperforming the Fixed Effects baseline by a large margin.

---

### Group 2 — Good (R²_Test 0.75–0.90)

| Indicator Code | RMSE_Test | R²_Test |
|---|---|---|
| GE.EST | 0.1313 | **0.9454** |
| IT.NET.USER.ZS | 1.9451 | **0.8411** |
| SG.GEN.PARL.ZS | 2.8960 | **0.9014** |
| SL.TLF.CACT.FM.ZS | 0.9811 | **0.9672** |
| PV.EST | 0.1016 | **0.7789** |

Solid performance. XGBoost handles the non-linearities in these indicators that the linear Fixed Effects model struggled with.

---

### Group 3 — Poor (R²_Test < 0)

| Indicator Code | RMSE_Test | R²_Test |
|---|---|---|
| NY.GDP.MKTP.KD.ZG | 4.3439 | **-1.5465** |
| SM.POP.NETM | 213808.29 | **-0.1006** |

GDP growth and net migration remain structurally difficult to forecast. Even XGBoost cannot capture exogenous shocks such as financial crises, geopolitical events, or sudden migration waves — these are driven by information not present in the lagged features.

---

### Comparison with Baseline (Fixed Effects)

XGBoost improves significantly over the Fixed Effects baseline for most indicators:

| Indicator Code | R²_Test (FE) | R²_Test (XGB) | Improvement |
|---|---|---|---|
| IT.NET.USER.ZS | -0.2478 | 0.8411 | +1.09 |
| SE.ENR.PRSC.FM.ZS | -1.6092 | 0.9173 | +2.53 |
| GE.EST | 0.6836 | 0.9454 | +0.26 |
| PV.EST | -0.0902 | 0.7789 | +0.87 |
| VA.EST | 0.6789 | 0.9675 | +0.29 |
| CC.EST | 0.8672 | 0.9880 | +0.12 |

The only cases where XGBoost also fails are `NY.GDP.MKTP.KD.ZG` and `SM.POP.NETM`,
confirming that these indicators are structurally unpredictable using lag-based features alone,
regardless of model complexity.

<div style="background-color:#EAF4EC; padding:16px; border-radius:10px;">
<h2 style="color:#2F6F4E; margin-bottom:5px; font-size:20px;">
Save data as .csv

</div>

In [12]:
out_path = "../../data/processed/XGB_2000_2030.csv"
df_xgb_full.to_csv(out_path, index=False)
print("Saved:", out_path, "| shape:", df_xgb_full.shape)

results_xgb.to_csv("../../data/processed/metrics_xgb.csv", index=False)

Saved: ../../data/processed/XGB_2000_2030.csv | shape: (12555, 7)
